For NUS:
    keep:
        "department"
        "description"
        "faculty"
        "moduleCode"
        "title"
        
    ignore:
        "gradingBasisDescription"
        "moduleCredit"
        "semesterData"
        "workload"


For NTU:
    keep:
        "code" 
        "title" 
        "description" 
        "departmentMaintaining"

    ignore:
        "academicUnits"
        "examSchedule" 
        "gradeType" 
        "prerequisites" 
        "notAvailableToProgramme" (eg not available to EEE)
        "url" 
        "details" 


For SUTD:
    keep:
        "code"
        "title" 
        "course_type" (core, electives, etc)
        "description" 
        "affiliations" (department)

    ignore:
        "terms" 
        "credits" 
        "url" 
        "description_found" (T/F)


extract skills from the descriptions

courses:
|university| department| faculty| code| title| course_type| description| skills|

jobs:
    keep: 
        "title"
        "description"
        "minimumYearsExperience"
        "shiftPattern"
        "skills" (keep everything in the skills section)
            "uuid": "11ff9f68afb6b8b5b8eda218d7c83a65",
            "confidence": null,
            "isKeySkill": null
        "otherRequirements"
        "ssocCode" (job industry/sector)
        "occupationId"
        "ssocVersion"
        "workingHours"
        "numberOfVacancies"
        "categories"
            "id"
            "category"
        "employmentTypes" 
            "id" 
            "employmentType" 
        "positionLevels"
            "id": 11,
            "position" 
        "ssecEqa" 
            (Singapore Standard Educational Classification (SSEC) Educational Qualification Assessment code, indicating the education level. 
            eg "69" typically corresponds to a Bachelor's Degree with Honours.)
        "ssecFos" 
            (Field of Study code, 
            eg "0212" corresponds to "Music and Performing Arts")
        "ssicCode" 
            (It classifies the type of industry/business sector the company operates in.)
            (SSOC = the job classification (Laboratory Technician, Finance Manager, etc.)
            (SSIC = the company/industry classification (Manufacturing, Education, etc.))
        "ssicCode2020" 
         "salary": {
                    "maximum": 3500,
                    "minimum": 2500,
                    "type": {
                    "id": 4,
                    "salaryType": "Monthly"
                    }
        "jobTitles"

    keep but tbc: (if all NA/nulls -> drop)
        "schemes" 
        "flexibleWorkArrangements"

    ignore:
        "uuid" (post id? user id?)
        "sourceCode" 
        "psdUrl"
        "status"
        "postedCompany" 
            "uen" 
                ("uen" stands for Unique Entity Number, a unique identifier assigned to all business entities (companies, organizations, partnerships) registered in Singapore. In the example, "52875677W" is the UEN for SYMPHONY MUSIC SCHOOL. It's used to uniquely identify and track companies in Singapore's business registry.)
            "description" (company description)
            "name" (company name)
        "employeeCount" 
        "companyUrl"
        "lastSyncDate": "2026-01-30T02:57:44.000Z",
        "_links"
        "logoFileName" 
        "logoUploadPath" 
        "responsiveEmployer": {
      "isResponsive": false
    }



# NUS

## Load Data

In [1]:
# import necessary libraries
import json5
import pandas as pd
from pathlib import Path

In [3]:
# load output NUSModsInfo.json file
nus_file_path = Path("../../data/NUSModsInfo.json")

with open(nus_file_path, "r", encoding="utf-8") as f:
    nus_data = json5.load(f)

In [4]:
# Extract relevant fields
nus_fields = ["department", "description", "faculty", "moduleCode", "title"]

nus_filtered_data = [
    {key: item.get(key) for key in nus_fields}
    for item in nus_data
]

In [5]:
# Create DataFrame
nus_df = pd.DataFrame(nus_filtered_data)

# Preview
nus_df.head()

,department,description,faculty,moduleCode,title
0,NUS Medicine Dean's Office,Leadership is fundamental to the success of in...,Yong Loo Lin Sch of Medicine,ABM5001,Leadership in Biomedicine
1,NUS Medicine Dean's Office,This course serves as a concept-based introduc...,Yong Loo Lin Sch of Medicine,ABM5002,Advanced Biostatistics for Research
2,NUS Medicine Dean's Office,This course will furnish students with a thoro...,Yong Loo Lin Sch of Medicine,ABM5003,Biomedical Innovation & Enterprise
3,NUS Medicine Dean's Office,This course encompasses research projects rele...,Yong Loo Lin Sch of Medicine,ABM5004,Capstone Project
4,NUS Medicine Dean's Office,Advanced immunological applications play impor...,Yong Loo Lin Sch of Medicine,ABM5101,Applied Immunology


## Data Cleaning

### Data Exploration

In [6]:
# check for missing values
print(nus_df.isnull().sum())

department     103
description      0
faculty          0
moduleCode       0
title            0
dtype: int64


In [7]:
nus_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20517 entries, 0 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   department   20414 non-null  str  
 1   description  20517 non-null  str  
 2   faculty      20517 non-null  str  
 3   moduleCode   20517 non-null  str  
 4   title        20517 non-null  str  
dtypes: str(5)
memory usage: 801.6 KB


In [8]:
# unique values in department and faculty
print("Unique departments:", nus_df["department"].nunique())
print("Unique faculties:", nus_df["faculty"].nunique()) 

Unique departments: 109
Unique faculties: 25


In [9]:
# get unique faculties 
unique_faculties = nus_df["faculty"].unique()
print("Unique faculties:", unique_faculties)

Unique faculties: <StringArray>
[     'Yong Loo Lin Sch of Medicine', 'College of Design and Engineering',
               'NUS Business School',           'Arts and Social Science',
                               'NUS',                         'Computing',
                               'Law',       'Cont and Lifelong Education',
                           'Science',     'Non-Faculty-based Departments',
                         'Dentistry',         'YST Conservatory of Music',
      'Multi Disciplinary Programme',                           'NUS-ISS',
               'Residential College',    'Temasek Defence Sys. Institute',
         'Risk Management Institute',                       'NUS College',
       'SSH School of Public Health',           'Duke-NUS Medical School',
               'NUS Graduate School',           'Logistics Inst-Asia Pac',
    'Mechanobiology Institute (MBI)',       'LKY School of Public Policy',
                  'Yale-NUS College']
Length: 25, dtype: str


In [10]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              5679
Law                                  2878
College of Design and Engineering    2823
Science                              1695
NUS Business School                  1642
Yale-NUS College                     1485
Computing                             541
Yong Loo Lin Sch of Medicine          517
Cont and Lifelong Education           425
Duke-NUS Medical School               416
LKY School of Public Policy           377
YST Conservatory of Music             373
Non-Faculty-based Departments         366
NUS College                           337
Residential College                   261
SSH School of Public Health           208
NUS                                   154
NUS-ISS                                82
NUS Graduate School                    65
Dentistry                              59
Temasek Defence Sys. Institute         51
Risk Management Institute              48
Multi Disciplinary Programme           15
Mechanobiology Institute (

### Filter for undergraduate courses only

In [11]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
nus_df = nus_df[nus_df['moduleCode'].str.match(r'^[A-Z]+[0-4]')]

nus_df

,department,description,faculty,moduleCode,title
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers
...,...,...,...,...,...
20512,Biological Sciences,In addition to having an academic science foun...,Science,ZB3312,Enhanced Undergraduate Professional Internship...
20513,Biological Sciences,In addition to having an academic science foun...,Science,ZB3313,Undergraduate Professional Internship Programm...
20514,Biological Sciences,This is a seminar-style course based on the li...,Science,ZB4171,Advanced Topics in Bioinformatics
20515,Biological Sciences,Not Available,Science,ZB4199,Honours Project in Computational Biology


In [12]:
# faculty level data distribution
faculty_counts = nus_df["faculty"].value_counts()
print(faculty_counts)

faculty
Arts and Social Science              4631
Yale-NUS College                     1485
College of Design and Engineering    1374
Science                              1172
NUS Business School                   796
Law                                   783
Cont and Lifelong Education           403
Computing                             350
YST Conservatory of Music             337
NUS College                           337
Non-Faculty-based Departments         333
Residential College                   261
Yong Loo Lin Sch of Medicine          174
NUS                                   122
SSH School of Public Health            61
Dentistry                              50
NUS-ISS                                19
Multi Disciplinary Programme           15
Duke-NUS Medical School                 3
Temasek Defence Sys. Institute          2
Name: count, dtype: int64


In [13]:
# filter out post graduate level courses: faculty is for post graduate level courses, which are not relevant to our analysis
target_faculties = [
    "Temasek Defence Sys. Institute",
     "Cont and Lifelong Education",
     "Duke-NUS Medical School"
]

filtered_nus_df = nus_df[~nus_df["faculty"].isin(target_faculties)]

### Handling NA Values

In [14]:
# count data with null description
null_description_count = filtered_nus_df["description"].isnull().sum()
print("Number of courses with null description:", null_description_count)

Number of courses with null description: 0


In [15]:
# Distribution of description data
description_counts = filtered_nus_df["description"].value_counts()
print(description_counts)

description
Not Available                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               3664
UTOP aims to train undergraduates to acquire and promulgate “teaching” skills. When pursuing such traineeship in teaching under UTOP, students can develop their teaching skills in a more systematic manner, and become be

some short descriptions such as: NIL, Exchange Course - YUS (1 unit), Not Applicable 
are not meaningful for analysis. we would remove those data

In [16]:
# Keep only rows where description has 10 or more words
filtered_nus_df = filtered_nus_df[filtered_nus_df['description'].str.split().str.len() >= 10]

In [17]:
filtered_nus_df.info()

<class 'pandas.DataFrame'>
Index: 8499 entries, 27 to 20516
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   department   8499 non-null   str  
 1   description  8499 non-null   str  
 2   faculty      8499 non-null   str  
 3   moduleCode   8499 non-null   str  
 4   title        8499 non-null   str  
dtypes: str(5)
memory usage: 398.4 KB


there is no more NA data in the dataset

## Data Transformation

In [18]:
filtered_nus_df["University"] = "NUS"

In [19]:
filtered_nus_df.head()

,department,description,faculty,moduleCode,title,University
27,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701,Accounting for Decision Makers,NUS
28,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701A,Accounting for Decision Makers,NUS
29,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701B,Accounting for Decision Makers,NUS
30,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701C,Accounting for Decision Makers,NUS
31,Accounting,The course provides an introduction to account...,NUS Business School,ACC1701D,Accounting for Decision Makers,NUS


# NTU

## Load Data

In [47]:
# load output ntuCourseInfo.json file
ntu_file_path = Path("../../data/ntuCourseInfo.json")

with open(ntu_file_path, "r", encoding="utf-8") as f:
    ntu_data = json5.load(f)

In [48]:
# Extract relevant fields
ntu_fields =  ["code", "title", "description", "departmentMaintaining"]

ntu_filtered_data = [
    {key: item.get(key) for key in ntu_fields}
    for item in ntu_data
]

In [49]:
# Create DataFrame
ntu_df = pd.DataFrame(ntu_filtered_data)

# Preview
ntu_df.head()

,code,title,description,departmentMaintaining
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE


## Data Cleaning

### Data Exploration

In [50]:
# check for missing values
print(ntu_df.isnull().sum())

code                      0
title                     0
description               0
departmentMaintaining    14
dtype: int64


In [51]:
ntu_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2117 entries, 0 to 2116
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   code                   2117 non-null   str  
 1   title                  2117 non-null   str  
 2   description            2117 non-null   str  
 3   departmentMaintaining  2103 non-null   str  
dtypes: str(4)
memory usage: 66.3 KB


In [52]:
# unique values in department
print("Unique departments:", ntu_df["departmentMaintaining"].nunique())

Unique departments: 47


In [53]:
# faculty level data distribution
ntu_dept_counts = ntu_df["departmentMaintaining"].value_counts()
print(ntu_dept_counts)

departmentMaintaining
BUS          144
ADM          144
CSC(CE)      129
NIE          119
SOH          111
CS            96
EEE           96
ELH(SOH)      74
MATH(SPS)     71
CHIN(SOH)     67
ME            64
BS            62
LMS(SOH)      57
PSY(SSS)      56
EESS(ASE)     54
HIST(SOH)     53
MAT           53
ECON(SSS)     51
CBE           49
PPGA(SSS)     49
PHY(SPS)      49
PHIL(SOH)     48
SOC(SSS)      41
CEE           39
CHEM(CBE)     37
ACC           36
CE            36
BIE(CBE)      28
REP           25
SSM(NIE)      22
MS(CEE)       21
ENE(CEE)      20
AERO(ME)      20
CMED(BS)      15
NTC           14
BMS(BS)       13
SSS            9
ICC            8
SPS            6
BSB(BS)        4
IEM(EEE)       4
ROBO(ME)       3
BACF(CE)       2
BMS            1
BSB            1
CAO            1
ACDA(CE)       1
Name: count, dtype: int64


### Filter for undergraduate courses only

In [54]:
# Keep only undergraduate level courses: codes where the first digit after letters is 0-4
ntu_df = ntu_df[ntu_df['code'].str.match(r'^[A-Z]+[0-4]')]

### Add department name

In [55]:
ntu_dept_file_path = Path("../../data/ntu_dept_mapping.xlsx")

ntu_dept_df = pd.read_excel(ntu_dept_file_path)

ntu_dept_df.head()

,short,department
0,ACC,Nanyang Business School (Accountancy)
1,ACDA(CE),Programme under Civil Engineering
2,ADM,"School of Art, Design and Media ..."
3,AERO(ME),Aerospace Engineering (MAE)
4,BACF(CE),Bachelor of Applied Computing in Finance (CE)


In [57]:
# add department mapping to ntu_df
ntu_df = ntu_df.merge(
    ntu_dept_df[["short", "department"]],
    left_on="departmentMaintaining",
    right_on="short",
    how="left"
).drop(columns=["short"])

In [59]:
ntu_df.rename(columns={"departmentMaintaining": "dept_code"}, inplace=True)

### Handling NA Values

In [65]:
# count data with null description
ntu_null_description_count = ntu_df["description"].isna().sum()
print("Number of courses with null description:", ntu_null_description_count)

Number of courses with null description: 0


In [62]:
# Distribution of description data
ntu_description_counts = ntu_df["description"].value_counts()
print(ntu_description_counts)

description
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [68]:
# Keep only rows where description has 10 or more words
filtered_ntu_df = ntu_df[ntu_df['description'].str.split().str.len() >= 10]

In [69]:
filtered_ntu_df.info()

<class 'pandas.DataFrame'>
Index: 1818 entries, 0 to 1851
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   code         1818 non-null   str  
 1   title        1818 non-null   str  
 2   description  1818 non-null   str  
 3   dept_code    1818 non-null   str  
 4   department   1818 non-null   str  
dtypes: str(5)
memory usage: 85.2 KB


## Data Transformation

In [70]:
filtered_ntu_df["University"] = "NTU"

In [72]:
filtered_ntu_df.head()

,code,title,description,dept_code,department,University
0,AAA08B,AAA08B FASHION & DESIGN: WEARABLE ART AS A SEC...,Develop process skills in expression and visua...,NIE,National Institute of Education,NTU
1,AAA08C,AAA08C EXPRESSIVE DRAWING: DEVELOPING PERSONAL...,Students will learn a brief history of express...,NIE,National Institute of Education,NTU
2,AAA08D,AAA08D ABSTRACT PAINTING: WHY IT'S HERE & HOW ...,Students will learn a brief history of abstrac...,NIE,National Institute of Education,NTU
3,AAA18D,AAA18D LIFE DRAWING,This course introduces drawing of the nude hum...,NIE,National Institute of Education,NTU
4,AAA18E,AAA18E DRAWING,This course introduces drawing at a basic leve...,NIE,National Institute of Education,NTU


# SUTD

## Load Data

In [77]:
# load output from sutdCourseDescriptions.json file
sutd_file_path = Path("../../data/sutdCourseDescriptions.json")

with open(sutd_file_path, "r", encoding="utf-8") as f:
    sutd_data = json5.load(f)

In [81]:
# Extract relevant fields
sutd_fields =  ["code", "title", "course_type", "description", "affiliations"]

sutd_filtered_data= [
    {key: item.get(key) for key in sutd_fields}
    for item in sutd_data
]

In [82]:
# Create DataFrame
sutd_df = pd.DataFrame(sutd_filtered_data)

sutd_df.head()

,code,title,course_type,description,affiliations
0,01.018,Design Thinking Project I,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT]
1,01.019,Design Thinking Project II,Freshmore core,"Design Thinking Projects I, II, and III provid...","[EPD, SMT]"
2,01.020,Design Thinking Project III,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT]
3,01.101,Technologies for Sustainable Global Health,Elective / Technical Elective,This course provides a multi-disciplinary appr...,"[EPD, SMT]"
4,01.102,Energy Systems and Management,Elective / Technical Elective,"Broadly, this course aims to acquaint students...",[ESD]


## Data Cleaning

### Data Exploration

In [80]:
# check for missing values
print(sutd_df.isnull().sum())

code           0
title          0
course_type    0
description    5
dtype: int64


### Add department name

In [84]:
AFFILIATION_NAMES = {
    "ASD": "Architecture and Sustainable Design",
    "DAI": "Design and Artificial Intelligence",
    "EPD": "Engineering Product Development",
    "ESD": "Engineering Systems and Design",
    "HASS": "Humanities, Arts and Social Sciences",
    "ISTD": "Information Systems Technology and Design",
    "SMT": "Science, Mathematics and Technology",
}

sutd_df["department"] = sutd_df["affiliations"].apply(
    lambda codes: [AFFILIATION_NAMES[c] for c in codes if c in AFFILIATION_NAMES]
)

### Handling NA Values

In [86]:
# count data with null description
sutd_null_description_count = sutd_df["description"].isna().sum()
print("Number of courses with null description:", sutd_null_description_count)

Number of courses with null description: 5


In [87]:
# Distribution of description data
sutd_description_counts = sutd_df["description"].value_counts()
print(sutd_description_counts)

description
Design Thinking Projects I, II, and III provide the Freshmore students the opportunity to learn and practice fundamental design thinking principles. Students are introduced to design thinking through a series of design-centric, interdisciplinary, multidisciplinary, hands-on projects and seminars, guided by a yearly general theme to ensure integrated pedagogy and progressive learning. Moreover, the projects and their solutions are expected to impact areas of growth at SUTD, such as healthcare, cities, aviation, and data science. 01.018 Design Thinking Project I in term 1 is a guided project, 01.019 Design Thinking Projects II in term 2 is integrated with 3.007 Design Thinking and Innovation course, while 01.020 Design Thinking Project III in term 3 allows the students with more scope for self-guidance. Ultimately, the courses are designed to equip the students with a series of design thinking, technical, contextual, organizational, leadership, scientific and technological sk

In [88]:
# remove courses with description less than 10 words and NA descriptions
filtered_sutd_df = sutd_df[sutd_df['description'].str.split().str.len() >= 10]

## Data Transformation

In [90]:
filtered_sutd_df["University"] = "SUTD"

In [91]:
filtered_sutd_df.head()

,code,title,course_type,description,affiliations,department,University
0,01.018,Design Thinking Project I,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT],"[Science, Mathematics and Technology]",SUTD
1,01.019,Design Thinking Project II,Freshmore core,"Design Thinking Projects I, II, and III provid...","[EPD, SMT]","[Engineering Product Development, Science, Mat...",SUTD
2,01.020,Design Thinking Project III,Freshmore core,"Design Thinking Projects I, II, and III provid...",[SMT],"[Science, Mathematics and Technology]",SUTD
3,01.101,Technologies for Sustainable Global Health,Elective / Technical Elective,This course provides a multi-disciplinary appr...,"[EPD, SMT]","[Engineering Product Development, Science, Mat...",SUTD
4,01.102,Energy Systems and Management,Elective / Technical Elective,"Broadly, this course aims to acquaint students...",[ESD],[Engineering Systems and Design],SUTD


# Jobs

## Load Data

In [2]:
from pathlib import Path
import json
import pandas as pd

def extract_job_jsons_to_df(job_folder="../../data/train/job") -> pd.DataFrame:
    job_folder = Path(job_folder)
    rows = []

    files = list(job_folder.rglob("*.json"))
    total_files = len(files)

    for i, file_path in enumerate(files, start=1):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            salary = data.get("salary", {}) or {}
            salary_type = salary.get("type", {}) or {}

            row = {
                "file_name": file_path.name,
                "title": data.get("title"),
                "description": data.get("description"),
                "minimumYearsExperience": data.get("minimumYearsExperience"),
                "shiftPattern": data.get("shiftPattern"),
                "otherRequirements": data.get("otherRequirements"),
                "ssocCode": data.get("ssocCode"),
                "occupationId": data.get("occupationId"),
                "ssocVersion": data.get("ssocVersion"),
                "workingHours": data.get("workingHours"),
                "numberOfVacancies": data.get("numberOfVacancies"),
                "ssecEqa": data.get("ssecEqa"),
                "ssecFos": data.get("ssecFos"),
                "ssicCode": data.get("ssicCode"),
                "ssicCode2020": data.get("ssicCode2020"),
                "jobTitles": data.get("jobTitles"),
                "skills": data.get("skills"),
                "categories": data.get("categories"),
                "employmentTypes": data.get("employmentTypes"),
                "positionLevels": data.get("positionLevels"),
                "salary_minimum": salary.get("minimum"),
                "salary_maximum": salary.get("maximum"),
                "salary_type_id": salary_type.get("id"),
                "salary_type": salary_type.get("salaryType"),
            }

            rows.append(row)

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

        if i % 1000 == 0:
            print(f"Processed {i}/{total_files} files ({(i/total_files)*100:.1f}%)")

    print(f"Done. Total processed: {total_files}")

    return pd.DataFrame(rows)

In [3]:
df_jobs = extract_job_jsons_to_df("../../data/train/job")
print(df_jobs.shape)
print(df_jobs.head())

Processed 1000/22720 files (4.4%)
Processed 2000/22720 files (8.8%)
Processed 3000/22720 files (13.2%)
Processed 4000/22720 files (17.6%)
Processed 5000/22720 files (22.0%)
Processed 6000/22720 files (26.4%)
Processed 7000/22720 files (30.8%)
Processed 8000/22720 files (35.2%)
Processed 9000/22720 files (39.6%)
Processed 10000/22720 files (44.0%)
Processed 11000/22720 files (48.4%)
Processed 12000/22720 files (52.8%)
Processed 13000/22720 files (57.2%)
Processed 14000/22720 files (61.6%)
Processed 15000/22720 files (66.0%)
Processed 16000/22720 files (70.4%)
Processed 17000/22720 files (74.8%)
Processed 18000/22720 files (79.2%)
Processed 19000/22720 files (83.6%)
Processed 20000/22720 files (88.0%)
Processed 21000/22720 files (92.4%)
Processed 22000/22720 files (96.8%)
Done. Total processed: 22720
(22720, 24)
               file_name                                              title  \
0  MCF-2026-0180015.json  Outdoor Sales Engineer (Own Car) | Petrochemic...   
1  MCF-2026-0182028.

In [103]:
df_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 22720 entries, 0 to 22719
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   file_name               22720 non-null  str   
 1   title                   22720 non-null  str   
 2   description             22720 non-null  str   
 3   minimumYearsExperience  22720 non-null  int64 
 4   shiftPattern            0 non-null      object
 5   otherRequirements       0 non-null      object
 6   ssocCode                22720 non-null  str   
 7   occupationId            22720 non-null  str   
 8   ssocVersion             22720 non-null  str   
 9   workingHours            0 non-null      object
 10  numberOfVacancies       22720 non-null  int64 
 11  ssecEqa                 22720 non-null  str   
 12  ssecFos                 15446 non-null  str   
 13  ssicCode                0 non-null      object
 14  ssicCode2020            0 non-null      object
 15  jobTitles    

In [4]:
df_jobs.head()

,file_name,title,description,minimumYearsExperience,shiftPattern,otherRequirements,ssocCode,occupationId,ssocVersion,workingHours,...,ssicCode2020,jobTitles,skills,categories,employmentTypes,positionLevels,salary_minimum,salary_maximum,salary_type_id,salary_type
0,MCF-2026-0180015.json,Outdoor Sales Engineer (Own Car) | Petrochemic...,<ul><li><p><strong>Role: Outdoor Sales Enginee...,1,None,None,24331,OCC003416,2020v2,None,...,None,[],"[{'skill': 'Project Bidding', 'uuid': '029f44d...","[{'id': 11, 'category': 'Engineering'}, {'id':...","[{'id': 7, 'employmentType': 'Permanent'}, {'i...","[{'id': 9, 'position': 'Executive'}]",3000,6000,4,Monthly
1,MCF-2026-0182028.json,IT Administrator (Coding) @ Changi (1 Year Ren...,<p><u><strong>Job Details:</strong></u></p><ul...,3,None,None,13304,OCC001994,2020v2,None,...,None,[],"[{'skill': 'Outlook', 'uuid': '038e648f69b23b2...","[{'id': 2, 'category': 'Admin / Secretarial'},...","[{'id': 3, 'employmentType': 'Contract'}, {'id...","[{'id': 8, 'position': 'Senior Executive'}]",4500,5900,4,Monthly
2,MCF-2025-1964969.json,Outsourced Finance Executive,<p><u><strong>Job Description</strong></u></p>...,2,None,None,43129,OCC008846,2020v2,None,...,None,[],"[{'skill': 'Accounts Payable', 'uuid': '1ed711...","[{'id': 1, 'category': 'Accounting / Auditing ...","[{'id': 8, 'employmentType': 'Full Time'}]","[{'id': 9, 'position': 'Executive'}]",3000,4000,4,Monthly
3,MCF-2026-0173197.json,Enterprise Architect (JD#10809),<p>Job Summary</p><p>An opportunity for an Ent...,11,None,None,25113,OCC013953,2020v2,None,...,None,[],"[{'skill': 'COBIT', 'uuid': '0afebee04b864ad4b...","[{'id': 21, 'category': 'Information Technolog...","[{'id': 7, 'employmentType': 'Permanent'}]","[{'id': 2, 'position': 'Middle Management'}]",12000,14000,4,Monthly
4,MCF-2025-1692194.json,L2 Preschool Malay Language Teacher,<p><strong>Job Description:</strong></p>\n<ul>...,1,None,None,36100,OCC006865,2020v2,None,...,None,[],"[{'skill': 'Childcare', 'uuid': '08370cd014507...","[{'id': 10, 'category': 'Education and Trainin...","[{'id': 7, 'employmentType': 'Permanent'}, {'i...","[{'id': 12, 'position': 'Fresh/entry level'}]",4000,5000,4,Monthly


In [5]:
df_jobs.head(2)

,file_name,title,description,minimumYearsExperience,shiftPattern,otherRequirements,ssocCode,occupationId,ssocVersion,workingHours,...,ssicCode2020,jobTitles,skills,categories,employmentTypes,positionLevels,salary_minimum,salary_maximum,salary_type_id,salary_type
0,MCF-2026-0180015.json,Outdoor Sales Engineer (Own Car) | Petrochemic...,<ul><li><p><strong>Role: Outdoor Sales Enginee...,1,None,None,24331,OCC003416,2020v2,None,...,None,[],"[{'skill': 'Project Bidding', 'uuid': '029f44d...","[{'id': 11, 'category': 'Engineering'}, {'id':...","[{'id': 7, 'employmentType': 'Permanent'}, {'i...","[{'id': 9, 'position': 'Executive'}]",3000,6000,4,Monthly
1,MCF-2026-0182028.json,IT Administrator (Coding) @ Changi (1 Year Ren...,<p><u><strong>Job Details:</strong></u></p><ul...,3,None,None,13304,OCC001994,2020v2,None,...,None,[],"[{'skill': 'Outlook', 'uuid': '038e648f69b23b2...","[{'id': 2, 'category': 'Admin / Secretarial'},...","[{'id': 3, 'employmentType': 'Contract'}, {'id...","[{'id': 8, 'position': 'Senior Executive'}]",4500,5900,4,Monthly


## Data Cleaning

We will only consider jobs that require minimumYearsExperience <= 1 (for freshgraduates)

In [10]:
# remove data with minimumYearsExperience > 1 
df_jobs = df_jobs[df_jobs["minimumYearsExperience"] <= 1]

In [16]:
df_jobs["minimumYearsExperience"].value_counts()

minimumYearsExperience
1    5368
0    4109
Name: count, dtype: int64

There is much less job data available, after filtering for minimumYearsExperience

Also some columns have 0 values, we will remove those columns

In [13]:
# remove columns with 0 values
columns_to_remove = ["shiftPattern", "otherRequirements", "ssicCode", "ssicCode2020"]
df_jobs = df_jobs.drop(columns=columns_to_remove)

In [6]:
# Clean positionLevels column 

df_jobs["position"] = df_jobs["positionLevels"].apply(
    lambda x: x[0].get("position") 
    if isinstance(x, list) and len(x) > 0 and isinstance(x[0], dict) 
    else None
)

In [27]:
df_jobs.loc[:, ["positionLevels", "position"]].head()

,positionLevels,position
0,"[{'id': 9, 'position': 'Executive'}]",Executive
4,"[{'id': 12, 'position': 'Fresh/entry level'}]",Fresh/entry level
5,"[{'id': 10, 'position': 'Junior Executive'}]",Junior Executive
8,"[{'id': 12, 'position': 'Fresh/entry level'}]",Fresh/entry level
11,"[{'id': 12, 'position': 'Fresh/entry level'}]",Fresh/entry level


In [29]:
# find all unique positions
df_jobs["position"].value_counts()

position
Fresh/entry level    3969
Non-executive        1739
Executive            1638
Junior Executive     1455
Professional          374
Senior Executive      122
Manager               117
Middle Management      37
Senior Management      26
Name: count, dtype: int64

In [32]:
df_jobs["jobTitles"].value_counts()

jobTitles
[]    9477
Name: count, dtype: int64

In [33]:
# remove jobTitles column since it is not informative for our analysis
df_jobs = df_jobs.drop(columns=["jobTitles"])

In [ ]:
# find all data with "Senior Executive" position
senior_executives = df_jobs[df_jobs["position"] == "Senior Executive"]
senior_executives

there are jobs with hourly pay but this is not indicated correctly in payment type

In [37]:
# find distribution of salary for salary_minimum 
df_jobs["salary_minimum"].describe()

count     9477.000000
mean      2813.350955
std       1175.608892
min          1.000000
25%       2200.000000
50%       2800.000000
75%       3200.000000
max      18000.000000
Name: salary_minimum, dtype: float64

In [42]:
# find all rows of data with salary less than 5% and more than 95% of the distribution
salary_min_5 = df_jobs["salary_minimum"].quantile(0.05)
salary_min_95 = df_jobs["salary_minimum"].quantile(0.95)
df_jobs["salary_minimum"].between(salary_min_5, salary_min_95).value_counts()  
    

salary_minimum
True     8568
False     909
Name: count, dtype: int64